# Titanic — End-to-End Data Science (Decode Labs–style portfolio)

**Dataset:** [Titanic](https://www.kaggle.com/c/titanic) passenger records  
**Goal:** Collect → clean → explore → visualize → model (logistic regression), with the **same pipeline** as the original notebook so reported metrics stay comparable.

> Tip for LinkedIn: run all cells, then screenshot the executive summary (missingness + dashboard + confusion matrix) for your post.

## Task map (internship rubric)

| # | Focus | What this notebook shows |
|---|--------|---------------------------|
| 1 | Data collection & understanding | Load CSV, schema, dtypes, size |
| 2 | Cleaning & preprocessing | Missing values, imputation, duplicates, encoding |
| 3 | EDA | Summary stats, distributions, outliers |
| 4 | Visualization | Styled Seaborn charts + one-page dashboard |
| 5 | Predictive model | Logistic regression, accuracy, report, confusion matrix |

**Bonus:** Survival by extracted passenger *Title* (exploratory only — does not change the model inputs above).

In [ ]:
# --- Imports ---
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Presentation theme (Seaborn + matplotlib) ---
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.05)
PALETTE = ['#2ecc71', '#e74c3c']  # survived green, not survived red
sns.set_palette(PALETTE)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

## Task 1 — Data collection & dataset understanding

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('titanic.csv')
df.head()

In [ ]:
print('Dataset shape (rows, columns):', df.shape)
print('\nColumns:', list(df.columns))
print('\nData types:')
print(df.dtypes)
print('\nInfo:')
df.info()

In [ ]:
df.describe()

## Task 2 — Data cleaning & preprocessing

Same decisions as the original workflow: median `Age`, mode `Embarked`, drop high-missing `Cabin`, drop duplicates, then label-encode `Sex` and `Embarked` for modeling.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=missing.values, y=missing.index, color='#c0392b', ax=ax)
ax.set_title('Missing values by column')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

print(missing)

In [ ]:
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)

print('Duplicate rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

# Human-readable snapshot for charts (before label encoding)
df_eda = df.copy()

label_encoder = LabelEncoder()
df['Sex'] = label_encoder.fit_transform(df['Sex'])
df['Embarked'] = label_encoder.fit_transform(df['Embarked'])

print('\nMissing values after cleaning:')
print(df.isnull().sum())
df.head()

## Task 3 — Exploratory data analysis (EDA)

In [ ]:
print(df.describe())

## Task 4 — Data visualization

Categorical plots use **`df_eda`** (string labels). The **correlation heatmap** uses the **encoded `df`**, matching the numeric representation used in the original notebook.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax0 = axes[0, 0]
sns.countplot(data=df_eda, x='Survived', ax=ax0, palette=PALETTE)
ax0.set_xticklabels(['Did not survive (0)', 'Survived (1)'])
ax0.set_title('Survival counts')

ax1 = axes[0, 1]
sns.countplot(data=df_eda, x='Sex', hue='Survived', ax=ax1, palette=PALETTE)
ax1.set_title('Survival by sex')
ax1.legend(title='Survived', labels=['No', 'Yes'])

ax2 = axes[1, 0]
sns.histplot(df_eda['Age'], bins=30, kde=True, color='#3498db', ax=ax2)
ax2.set_title('Age distribution')

ax3 = axes[1, 1]
sns.countplot(data=df_eda, x='Pclass', hue='Survived', ax=ax3, palette=PALETTE)
ax3.set_title('Passenger class vs survival')
ax3.legend(title='Survived', labels=['No', 'Yes'])

plt.suptitle('Titanic — EDA dashboard', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation heatmap (encoded features)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(x=df_eda['Fare'], color='#9b59b6', ax=ax)
ax.set_title('Fare distribution (outliers visible in the long right tail)')
plt.tight_layout()
plt.show()

## Task 5 — Predictive model (logistic regression)

Same feature matrix as before: drop `PassengerId`, `Name`, `Ticket`; predict `Survived` with `train_test_split(..., random_state=42)`.

In [ ]:
df_model = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)
X = df_model.drop('Survived', axis=1)
y = df_model['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print('Model accuracy:', round(accuracy, 4))
print('\nClassification report:')
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    square=True,
    cbar=False,
    ax=ax,
    annot_kws={'size': 12},
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticklabels(['Died (0)', 'Survived (1)'])
ax.set_yticklabels(['Died (0)', 'Survived (1)'], rotation=0)
ax.set_title('Confusion matrix')
plt.tight_layout()
plt.show()

## Bonus — “So what?” view (does not change the model)

**Question:** Did social title (Mr / Mrs / Master / …) line up with survival, before any ML?

We parse `Name` on `df_eda` only for this exploratory chart.

In [ ]:
def extract_title(name):
    if pd.isna(name):
        return 'Unknown'
    part = name.split(',')[1].split('.')[0].strip()
    return part

t = df_eda.copy()
t['Title'] = t['Name'].apply(extract_title)

title_order = t['Title'].value_counts().head(8).index
t_top = t[t['Title'].isin(title_order)]

plt.figure(figsize=(11, 5))
ax = sns.barplot(
    data=t_top,
    x='Title',
    y='Survived',
    hue='Sex',
    palette='muted',
    errorbar=None,
)
ax.set_xlabel('Title (from name)')
ax.set_ylabel('Mean survival rate')
ax.set_title('Bonus: survival rate by title (exploratory)')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---

### LinkedIn caption ideas (short)

- Walkthrough of 5 DS tasks on Titanic: cleaning, EDA, visuals, and logistic regression — same split/seed for reproducible accuracy.
- Bonus chart: survival vs extracted passenger title — quick storytelling layer on top of the baseline model.

**Decode Labs:** [decodelabs.tech](https://www.decodelabs.tech)